In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

In [2]:
import boto3
import pandas as pd
from io import StringIO

s3 = boto3.client("s3", region_name="us-east-1")
bucket = "food-waste-project"

def load_csv(key, **kwargs):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(StringIO(obj["Body"].read().decode("utf-8")), **kwargs)

products = load_csv("raw/products.csv", usecols=["product_id", "product_name", "aisle_id", "department_id"])
aisles = load_csv("raw/aisles.csv", usecols=["aisle_id", "aisle"])
departments = load_csv("raw/departments.csv", usecols=["department_id", "department"])

print("Products shape:", products.shape)
print("Aisles shape:", aisles.shape)
print("Departments shape:", departments.shape)

Products shape: (49688, 4)
Aisles shape: (134, 2)
Departments shape: (21, 2)


In [3]:
# Merge product metadata
df = products.merge(aisles, on="aisle_id", how="left")
df = df.merge(departments, on="department_id", how="left")

print("Shape:", df.shape)
print(df.head())


Shape: (49688, 6)
   product_id                                       product_name  aisle_id  \
0           1                         Chocolate Sandwich Cookies        61   
1           2                                   All-Seasons Salt       104   
2           3               Robust Golden Unsweetened Oolong Tea        94   
3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...        38   
4           5                          Green Chile Anytime Sauce         5   

   department_id                       aisle department  
0             19               cookies cakes     snacks  
1             13           spices seasonings     pantry  
2              7                         tea  beverages  
3              1                frozen meals     frozen  
4             13  marinades meat preparation     pantry  


In [4]:
#basic cleaning
df["product_name"] = df["product_name"].fillna("").astype(str)
df["aisle"] = df["aisle"].fillna("").astype(str)
df["department"] = df["department"].fillna("").astype(str)

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["product_name_clean"] = df["product_name"].apply(clean_text)
df["aisle_clean"] = df["aisle"].apply(clean_text)
df["department_clean"] = df["department"].apply(clean_text)

In [5]:
#filter to food only
food_departments = [
    "produce",
    "dairy eggs",
    "meat seafood",
    "frozen",
    "pantry",
    "bakery",
    "snacks",
    "beverages",
    "dry goods pasta",
    "canned goods",
    "breakfast",
    "deli",
    "international",
    "alcohol"
]

df = df[df["department"].isin(food_departments)].copy()

print("New shape:", df.shape)

New shape: (36143, 9)


In [6]:
#create target label
def assign_waste_category(row):
    """
    Assign a broad waste/perishability category using aisle, department,
    and product name. You can adjust these rules later if your team wants.
    """
    name = row["product_name_clean"]
    aisle = row["aisle_clean"]
    dept = row["department_clean"]

    text = f"{name} {aisle} {dept}"

    # Frozen
    if any(word in text for word in [
        "frozen", "ice cream", "frozen meals", "frozen dessert", "frozen pizza"
    ]):
        return "frozen"

    # Beverage
    if any(word in text for word in [
        "water", "sparkling water", "juice", "soda", "coffee", "tea",
        "beverage", "drink", "sports drink", "energy drink"
    ]):
        return "beverage"

    # Snack
    if any(word in text for word in [
        "chips", "cookie", "cookies", "cracker", "crackers", "popcorn",
        "candy", "chocolate", "granola bar", "snack", "pretzel", "trail mix"
    ]):
        return "snack"

    if any(word in text for word in ["canned", "jarred"]):
        return "pantry"

# Pantry override (important fix)
    if any(word in text for word in [
    "sauce", "syrup", "oil", "seasoning", "spice",
    "pasta", "rice", "grain", "quinoa", "dry", "mix"
  ]):
     return "pantry"
    # Fresh / highly perishable
    if any(word in text for word in [
        "fresh fruits", "fresh vegetables", "produce", "fruit", "vegetable",
        "salad", "herbs", "milk", "yogurt", "eggs", "cream", "cheese",
        "deli", "meat", "seafood", "poultry", "fresh pasta", "tofu"
    ]):
        return "fresh"

    # Pantry / shelf-stable
    if any(word in text for word in [
        "pasta", "rice", "canned", "soup", "beans", "baking", "flour",
        "sugar", "oil", "vinegar", "spices", "cereal", "oatmeal",
        "nuts", "dried fruit", "grains", "condiments", "sauce"
    ]):
        return "pantry"

    # Use department / aisle fallback rules
    if dept in ["produce", "dairy eggs", "deli", "meat seafood"]:
        return "fresh"

    if dept in ["frozen"]:
        return "frozen"

    if dept in ["beverages", "alcohol"]:
        return "beverage"

    if dept in ["snacks", "candy"]:
        return "snack"

    if dept in ["dry goods pasta", "canned goods", "pantry", "breakfast", "bakery"]:
        return "pantry"

    # Default fallback
    return "pantry"

df["waste_category"] = df.apply(assign_waste_category, axis=1)

print("\nCategory counts:")
print(df["waste_category"].value_counts())

# inspect examples
for cat in df["waste_category"].unique():
    print(f"\nExamples for {cat}:")
    print(df.loc[df["waste_category"] == cat, ["product_name", "aisle", "department"]].head(5))


Category counts:
waste_category
pantry      11421
fresh        7640
snack        6974
beverage     5989
frozen       4119
Name: count, dtype: int64

Examples for snack:
                                   product_name                aisle  \
0                    Chocolate Sandwich Cookies        cookies cakes   
24      Salted Caramel Lean Protein & Fiber Bar  energy granola bars   
31                Nacho Cheese White Bean Chips       chips pretzels   
40  Organic Sourdough Einkorn Crackers Rosemary             crackers   
46   Onion Flavor Organic Roasted Seaweed Snack          asian foods   

       department  
0          snacks  
24         snacks  
31         snacks  
40         snacks  
46  international  

Examples for pantry:
                                         product_name  \
1                                    All-Seasons Salt   
4                           Green Chile Anytime Sauce   
18   Gluten Free Quinoa Three Cheese & Mushroom Blend   
27                         

In [7]:
df["waste_category"] = df.apply(assign_waste_category, axis=1)
df["waste_category"].value_counts()

waste_category
pantry      11421
fresh        7640
snack        6974
beverage     5989
frozen       4119
Name: count, dtype: int64

In [8]:
#define features + target
X = df[["product_name_clean", "aisle_clean", "department_clean"]]
y = df["waste_category"]

In [9]:
#train/test split
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index,
    test_size=0.2,
    random_state=2026,
    stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (28914, 3)
Test shape: (7229, 3)


In [10]:
#preprocessing + model
text_features = "product_name_clean"
categorical_features = ["aisle_clean", "department_clean"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            TfidfVectorizer(
                max_features=5000,
                ngram_range=(1, 2),
                stop_words="english"
            ),
            text_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000))
])

In [11]:
#train model
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('text', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transform

In [12]:
#evaluate
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("Weighted F1:", f1_score(y_test, y_pred, average="weighted"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=sorted(y.unique())),
    index=sorted(y.unique()),
    columns=sorted(y.unique())
)
print(cm)


Accuracy: 0.9745469636187578
Weighted F1: 0.9746201943925678

Classification Report:
              precision    recall  f1-score   support

    beverage       1.00      0.96      0.98      1198
       fresh       0.95      0.97      0.96      1528
      frozen       1.00      0.99      1.00       824
      pantry       0.96      0.98      0.97      2284
       snack       0.99      0.97      0.98      1395

    accuracy                           0.97      7229
   macro avg       0.98      0.97      0.98      7229
weighted avg       0.97      0.97      0.97      7229


Confusion Matrix:
          beverage  fresh  frozen  pantry  snack
beverage      1154     18       0      16     10
fresh            2   1477       0      49      0
frozen           0      0     819       5      0
pantry           2     36       0    2246      0
snack            0     21       0      25   1349


In [13]:
#generate predictions for all products
df["predicted_waste_category"] = model.predict(X)

# If you want probabilities/confidence:
proba = model.predict_proba(X)
class_labels = model.classes_

proba_df = pd.DataFrame(proba, columns=[f"prob_{c}" for c in class_labels])

# max probability as confidence
df["prediction_confidence"] = proba.max(axis=1)

In [14]:
#export outputs
#Full product-level predictions
predictions_df = pd.concat([
    df[[
        "product_id",
        "product_name",
        "aisle",
        "department",
        "waste_category",
        "predicted_waste_category",
        "prediction_confidence"
    ]].reset_index(drop=True),
    proba_df.reset_index(drop=True)
], axis=1)

predictions_df.to_csv("waste_category_predictions.csv", index=False)

In [15]:
#SAVE CLEANED LABELED PRODUCT TABLE TOO
df.to_csv("instacart_waste_category_labeled.csv", index=False)

print("\nSaved files:")
print("- waste_category_predictions.csv")
print("- instacart_waste_category_labeled.csv")


Saved files:
- waste_category_predictions.csv
- instacart_waste_category_labeled.csv


In [16]:
#Sanity check
print("\nSample predictions:")
print(predictions_df.head(10))


Sample predictions:
   product_id                                       product_name  \
0           1                         Chocolate Sandwich Cookies   
1           2                                   All-Seasons Salt   
2           3               Robust Golden Unsweetened Oolong Tea   
3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...   
4           5                          Green Chile Anytime Sauce   
5           7                     Pure Coconut Water With Orange   
6           8                  Cut Russet Potatoes Steam N' Mash   
7           9                  Light Strawberry Blueberry Yogurt   
8          10     Sparkling Orange Juice & Prickly Pear Beverage   
9          11                                  Peach Mango Juice   

                           aisle  department waste_category  \
0                  cookies cakes      snacks          snack   
1              spices seasonings      pantry         pantry   
2                            tea   bevera

In [17]:
# export to aws
rds_df = predictions_df[[
    "product_id",
    "product_name",
    "predicted_waste_category",
    "prediction_confidence"
]].copy()  

#ADD THIS HERE (before saving)
risk_map = {
    "fresh": "high",
    "snack": "medium",
    "beverage": "medium",
    "pantry": "low",
    "frozen": "low"
}

rds_df["waste_risk"] = rds_df["predicted_waste_category"].map(risk_map)

# now save
rds_df.to_csv("waste_predictions_rds.csv", index=False)

In [18]:
import io

csv_buffer = io.StringIO()
rds_df.to_csv(csv_buffer, index=False)

s3.put_object(
    Bucket=bucket,
    Key="predictions/waste_predictions_rds.csv",
    Body=csv_buffer.getvalue()
)

print("Predictions uploaded to S3!")

Predictions uploaded to S3!


In [19]:
import joblib


# Save model locally first
joblib.dump(model, "waste_model.joblib")

# Upload to S3
s3.upload_file(
    "waste_model.joblib",
    "food-waste-project",
    "model/waste_model.joblib"
)

print("Model saved to S3!")

Model saved to S3!


In [20]:
inference_code = """
import joblib
import os
import pandas as pd

def model_fn(model_dir):
    model = joblib.load(os.path.join(model_dir, "waste_model.joblib"))
    return model

def input_fn(request_body, request_content_type):
    if request_content_type == "application/json":
        import json
        data = json.loads(request_body)
        return pd.DataFrame(data)
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    predictions = model.predict(input_data)
    probabilities = model.predict_proba(input_data).max(axis=1)
    
    result = pd.DataFrame({
        "predicted_waste_category": predictions,
        "confidence": probabilities
    })
    return result

def output_fn(prediction, response_content_type):
    return prediction.to_json(orient="records")
"""

with open("inference.py", "w") as f:
    f.write(inference_code)

print("inference.py created!")

inference.py created!


In [21]:
# Connect to RDS

In [22]:
!pip install pymysql -q

import pymysql

conn = pymysql.connect(
    host="food-waste-db.cxkacdkaxalv.us-east-1.rds.amazonaws.com",
    user="admin",
    password="ChurnLab2026!",
    port=3306
)

cursor = conn.cursor()
print("Connected!")

Connected!
